# Sentiment Analysis Summarizer

**Purpose:** Analyze and summarize sentiment patterns from Twitter data using RoBERTa.

**Key Features:**
- Fine-tuned RoBERTa model for sentiment classification
- Temporal trend analysis and aggregation
- Interactive visualizations
- Geographic sentiment heatmaps
- Statistical insights and summaries

---

**Quick Start:** Run cells sequentially. Training takes ~3 hours on 10K samples (CPU). Visualization cells require `nbformat` package.

In [ ]:
# Load dataset from CSV and prepare splits
from datasets import load_dataset

csv_path = "datasets/twitter_sentiment_data_CLEANED.csv"
dataset = load_dataset("csv", data_files={"train": csv_path}, split="train")

# If your CSV has columns like 'text' and 'label' adjust here
text_column = "text"
label_column = "label"

# Create validation split (10%)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]

{'sentiment': 1,
 'message': 'The sea floor is sinking under the weight of climate change https://t.co/R9Uhnjfg7G',
 'tweetid': 954625951685578752}

In [2]:
# Install training dependencies
import sys
!{sys.executable} -m pip install -U datasets scikit-learn

In [3]:
# Optional: install required packages (uncomment if needed)
## Note: Running installs from the notebook may require internet access.
## On Windows cmd, you can also install via terminal:
## pip install --upgrade pip
## pip install transformers torch sentencepiece

# If you prefer inline install, uncomment the following:
# import sys
# !{sys.executable} -m pip install -U pip
# !{sys.executable} -m pip install transformers torch sentencepiece

In [4]:
# Fix notebook progress bars and optional HF Xet support
# import sys
# !{sys.executable} -m pip install --upgrade ipywidgets jupyter jupyterlab notebook
# Optional: speed up Hugging Face downloads
# !{sys.executable} -m pip install "huggingface_hub[hf_xet]"
# Optional: silence symlink warning without enabling Developer Mode
# import os
# os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [5]:
# Initialize RoBERTa sentiment pipeline using CardiffNLP model
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
import torch

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Build a TextClassificationPipeline
sentiment_pipeline = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    framework="pt",
    device=0 if torch.cuda.is_available() else -1
)

labels = model.config.id2label
labels

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu


{0: 'negative', 1: 'neutral', 2: 'positive'}

In [6]:
# Quick test
texts = [
    "I love this product!",
    "This is the worst experience ever.",
    "It's okay, nothing special.",
    "I hate you so much!",
    "I adore you so much!"
]

results = sentiment_pipeline(texts, top_k=1)
for text, res in zip(texts, results):
    print(f"{text} -> {res[0]['label']} ({res[0]['score']:.3f})")

I love this product! -> positive (0.985)
This is the worst experience ever. -> negative (0.945)
It's okay, nothing special. -> neutral (0.599)
I hate you so much! -> negative (0.935)
I adore you so much! -> positive (0.984)


---

## Phase 1: Data Loading & Preparation

Load the labeled dataset with all required features: text, sentiment_label, timestamp, platform, and geolocation.

In [7]:
# Load labeled dataset with all features
import pandas as pd
from datasets import Dataset, DatasetDict

# Load your pre-processed and labeled data
# Using 10K sample for faster training (~2-3 hours instead of 12)
csv_path = r"10k_sample.csv"

df = pd.read_csv(csv_path)

# Print actual columns to diagnose issues
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Auto-detect sentiment column name (flexible column naming)
sentiment_col = None
for possible_name in ['sentiment_label', 'sentiment', 'label', 'Sentiment', 'target']:
    if possible_name in df.columns:
        sentiment_col = possible_name
        break

if sentiment_col is None:
    raise ValueError(f"Could not find sentiment column. Available columns: {df.columns.tolist()}")

print(f"\nUsing '{sentiment_col}' as sentiment column")

# Auto-detect text column name
text_col = None
for possible_name in ['text', 'message', 'tweet', 'content', 'Message']:
    if possible_name in df.columns:
        text_col = possible_name
        break

if text_col is None:
    raise ValueError(f"Could not find text column. Available columns: {df.columns.tolist()}")

print(f"Using '{text_col}' as text column")

# Standardize column names
if sentiment_col != 'sentiment_label':
    df['sentiment_label'] = df[sentiment_col]

if text_col != 'text':
    df['text'] = df[text_col]

# Check label distribution BEFORE conversion
print(f"\nOriginal Sentiment Distribution:")
print(df['sentiment_label'].value_counts())

# Convert numeric sentiments to strings if needed
# Common mappings: -1=Negative, 0=Neutral, 1=Positive, 2=Positive
if df['sentiment_label'].dtype in ['int64', 'float64']:
    print("\nDetected numeric sentiments. Converting to strings...")
    numeric_to_string = {
        -1: 'Negative',
        0: 'Neutral',
        1: 'Positive',
        2: 'Positive'  # Sometimes datasets use 2 for positive
    }
    df['sentiment_label'] = df['sentiment_label'].map(numeric_to_string)
    
    print(f"\nAfter conversion:")
    print(df['sentiment_label'].value_counts())

# Convert timestamps to datetime if timestamp column exists
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    print(f"\nDate Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
else:
    print("\nWarning: No 'timestamp' column found. Creating dummy timestamps.")
    df['timestamp'] = pd.date_range(start='2024-01-01', periods=len(df), freq='H')

# Ensure other columns exist or create defaults
if 'platform' not in df.columns:
    print("Warning: No 'platform' column found. Creating default.")
    df['platform'] = 'Twitter'

if 'geolocation' not in df.columns:
    print("Warning: No 'geolocation' column found. Creating default.")
    df['geolocation'] = 'Unknown'

print(f"\n✓ Data loading complete. Ready for label encoding.")


Dataset shape: (8818, 3)
Columns: ['sentiment', 'message', 'tweetid']

First few rows:
   sentiment                                            message  \
0         -1  @tiniebeany climate change is an interesting h...   
1          1  RT @NatGeoChannel: Watch #BeforeTheFlood right...   
2          1  Fabulous! Leonardo #DiCaprio's film on #climat...   
3          1  RT @Mick_Fanning: Just watched this amazing do...   
4          2  RT @cnalive: Pranita Biswasi, a Lutheran from ...   

              tweetid  
0  792927353886371840  
1  793124211518832641  
2  793124402388832256  
3  793124635873275904  
4  793125156185137153  

Using 'sentiment' as sentiment column
Using 'message' as text column

Original Sentiment Distribution:
sentiment_label
 1    5686
 2    1450
 0    1171
-1     511
Name: count, dtype: int64

Detected numeric sentiments. Converting to strings...

After conversion:
sentiment_label
Positive    7136
Neutral     1171
Negative     511
Name: count, dtype: int64


✓ Data 

C:\Users\codes\AppData\Local\Temp\ipykernel_21324\644892679.py:73: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['timestamp'] = pd.date_range(start='2024-01-01', periods=len(df), freq='H')


## Phase 2: Label Encoding & Dataset Preparation

Map sentiment labels to numeric IDs and prepare train/validation splits.

In [8]:
# Create label mapping: Positive=2, Neutral=1, Negative=0
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
id2label = {v: k for k, v in label_map.items()}

# Encode labels (sentiment_label column now exists from previous cell)
df['label'] = df['sentiment_label'].map(label_map)

# Verify no missing labels after mapping
missing_count = df['label'].isna().sum()
print(f"Missing labels after mapping: {missing_count}")

if missing_count > 0:
    print(f"\nWarning: Found {missing_count} unmapped labels. These will be removed.")
    print(f"Unmapped values: {df[df['label'].isna()]['sentiment_label'].unique()}")
    df = df.dropna(subset=['label'])

print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

# Create train/validation/test splits (70/15/15)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"\nSplit sizes:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'label', 'timestamp', 'platform', 'geolocation']].reset_index(drop=True))

print(f"\n✓ Datasets created successfully")

Missing labels after mapping: 0

Label distribution:
label
0     511
1    1171
2    7136
Name: count, dtype: int64

Split sizes:
Train: 6172
Validation: 1323
Test: 1323

✓ Datasets created successfully


## Phase 3: Model Fine-Tuning Setup

Configure RoBERTa tokenizer and prepare datasets for training.

In [9]:
# Initialize RoBERTa tokenizer and tokenize datasets
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding=False)

# Tokenize all datasets
train_tokenized = train_dataset.map(preprocess_function, batched=True)
val_tokenized = val_dataset.map(preprocess_function, batched=True)
test_tokenized = test_dataset.map(preprocess_function, batched=True)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"Tokenized train samples: {len(train_tokenized)}")

Map:   0%|          | 0/6172 [00:00<?, ? examples/s]

Map:   0%|          | 0/1323 [00:00<?, ? examples/s]

Map:   0%|          | 0/1323 [00:00<?, ? examples/s]

Device: CPU
Tokenized train samples: 6172


## Phase 4: Model Fine-Tuning & Training

Configure model architecture, training arguments, and execute fine-tuning with comprehensive metrics.

In [10]:
# Install training dependencies (accelerate required for Trainer)
# IMPORTANT: After running this cell, you may need to restart the kernel
import sys
!{sys.executable} -m pip install -q accelerate>=0.26.0

# Force reload transformers to pick up newly installed accelerate
import importlib
import transformers
importlib.reload(transformers)

print("✓ Accelerate installed. If you still see errors, restart the kernel.")

✓ Accelerate installed. If you still see errors, restart the kernel.


In [11]:
# Configure RoBERTa model and training pipeline
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# Initialize model with label mappings
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label_map
)

# Define comprehensive evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "f1_per_class": f1_score(labels, predictions, average=None).tolist()
    }

# Training configuration optimized for sentiment analysis
training_args = TrainingArguments(
    output_dir="./models/roberta-environmental-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # Mixed precision if GPU available
    report_to="none",  # Disable wandb/tensorboard for now
)

# Initialize Trainer with early stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting model fine-tuning...")
print(f"Training samples: {len(train_tokenized)}")
print(f"Validation samples: {len(val_tokenized)}")
print(f"Device: {training_args.device}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting model fine-tuning...
Training samples: 6172
Validation samples: 1323
Device: cpu


C:\Users\codes\AppData\Local\Temp\ipykernel_21324\1929521756.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
# Execute training
train_result = trainer.train()

# Evaluate on validation set
val_metrics = trainer.evaluate()
print("\n" + "="*50)
print("VALIDATION METRICS")
print("="*50)
for key, value in val_metrics.items():
    print(f"{key}: {value}")

# Save the best model
model_save_path = "./models/roberta-environmental-sentiment-best"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"\n✓ Model saved to: {model_save_path}")

c:\Users\codes\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Phase 5: Model Evaluation on Test Set

Generate predictions and detailed performance analysis.

In [ ]:
# Generate predictions on test set
predictions = trainer.predict(test_tokenized)
pred_labels = np.argmax(predictions.predictions, axis=-1)
pred_probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()

# Get true labels
true_labels = predictions.label_ids

# Compute test metrics
test_accuracy = accuracy_score(true_labels, pred_labels)
test_f1_macro = f1_score(true_labels, pred_labels, average="macro")
test_f1_weighted = f1_score(true_labels, pred_labels, average="weighted")

print("="*50)
print("TEST SET METRICS")
print("="*50)
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1 Macro: {test_f1_macro:.4f}")
print(f"F1 Weighted: {test_f1_weighted:.4f}")

print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(
    true_labels, 
    pred_labels, 
    target_names=["Negative", "Neutral", "Positive"],
    digits=4
))

print("="*50)
print("CONFUSION MATRIX")
print("="*50)
cm = confusion_matrix(true_labels, pred_labels)
print("           Predicted")
print("           Neg  Neu  Pos")
for i, label in enumerate(["Negative", "Neutral", "Positive"]):
    print(f"Actual {label:8s} {cm[i]}")

c:\Users\codes\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TEST SET METRICS
Accuracy: 0.8828
F1 Macro: 0.7434
F1 Weighted: 0.8792

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Negative     0.6506    0.7013    0.6750        77
     Neutral     0.6950    0.5568    0.6183       176
    Positive     0.9245    0.9495    0.9368      1070

    accuracy                         0.8828      1323
   macro avg     0.7567    0.7359    0.7434      1323
weighted avg     0.8780    0.8828    0.8792      1323

CONFUSION MATRIX
           Predicted
           Neg  Neu  Pos
Actual Negative [54  7 16]
Actual Neutral  [11 98 67]
Actual Positive [  18   36 1016]


## Phase 6: Prepare Predictions DataFrame

Create enriched dataset with predictions, probabilities, and metadata for temporal analysis.

In [ ]:
# Create comprehensive results dataframe
results_df = test_df.copy()
results_df['predicted_label'] = pred_labels
results_df['predicted_sentiment'] = results_df['predicted_label'].map(id2label)
results_df['prob_negative'] = pred_probs[:, 0]
results_df['prob_neutral'] = pred_probs[:, 1]
results_df['prob_positive'] = pred_probs[:, 2]
results_df['confidence'] = pred_probs.max(axis=1)

# Calculate sentiment score (-1 to +1 scale)
results_df['sentiment_score'] = (
    results_df['prob_positive'] - results_df['prob_negative']
)

# Ensure timestamp is datetime
results_df['timestamp'] = pd.to_datetime(results_df['timestamp'])

print(f"Results DataFrame shape: {results_df.shape}")
print(f"\nSample predictions:")
print(results_df[['text', 'sentiment_label', 'predicted_sentiment', 'sentiment_score', 'confidence']].head(10))

# Save predictions for future analysis
results_df.to_csv("predictions_with_metadata.csv", index=False)
print("\n✓ Predictions saved to: predictions_with_metadata.csv")

Results DataFrame shape: (1323, 16)

Sample predictions:
                                                   text sentiment_label  \
6872  RT @vbs269: It's currently 70 degrees and coul...        Positive   
2192  RT @ajplus: Leo takes on climate change skepti...        Positive   
3163  I'm still confused on how y'all voted for a ma...        Positive   
7274  RT @BetteMidler: We are so fucked. Trump picks...        Positive   
6068  @YahooNews Why not spend your money on climate...        Positive   
3591  Gosh, It's cold in the morning and hot in the ...        Negative   
7549  “Trump lines up staff to avoid international a...        Positive   
6844  RT @CounterMoonbat: The people who predicted p...        Negative   
6694  RT @colbertlateshow: Donald Trump called globa...        Positive   
5862  RT @StephenSchlegel: she's thinking about how ...        Positive   

     predicted_sentiment  sentiment_score  confidence  
6872            Positive         0.998438    0.999011  
2192 

---

## Phase 7: Temporal Aggregation & Trend Analysis

Aggregate sentiment scores by time windows and platform to identify patterns.

In [ ]:
# Temporal aggregation functions
def aggregate_by_time(df, freq='W', score_col='sentiment_score'):
    """
    Aggregate sentiment by time period
    freq: 'D' (daily), 'W' (weekly), 'M' (monthly)
    """
    df = df.copy()
    df = df.set_index('timestamp')
    
    aggregated = df.groupby(pd.Grouper(freq=freq)).agg({
        score_col: ['mean', 'std', 'count'],
        'prob_positive': 'mean',
        'prob_neutral': 'mean',
        'prob_negative': 'mean',
        'confidence': 'mean'
    })
    
    aggregated.columns = ['_'.join(col).strip() for col in aggregated.columns.values]
    aggregated = aggregated.reset_index()
    
    return aggregated

# Weekly aggregation
weekly_sentiment = aggregate_by_time(results_df, freq='W')
print("Weekly Sentiment Aggregation:")
print(weekly_sentiment.head(10))
print(f"\nTotal weeks: {len(weekly_sentiment)}")

# Monthly aggregation
monthly_sentiment = aggregate_by_time(results_df, freq='M')
print("\n" + "="*50)
print("Monthly Sentiment Aggregation:")
print(monthly_sentiment.head())

# Aggregate by platform
platform_sentiment = results_df.groupby('platform').agg({
    'sentiment_score': ['mean', 'std', 'count'],
    'prob_positive': 'mean',
    'prob_neutral': 'mean',
    'prob_negative': 'mean'
}).reset_index()

platform_sentiment.columns = ['_'.join(col).strip() if col[1] else col[0] 
                               for col in platform_sentiment.columns.values]
print("\n" + "="*50)
print("Platform-wise Sentiment:")
print(platform_sentiment)

Weekly Sentiment Aggregation:
   timestamp  sentiment_score_mean  sentiment_score_std  \
0 2024-01-07              0.799527             0.389183   
1 2024-01-14              0.777816             0.417810   
2 2024-01-21              0.657249             0.465201   
3 2024-01-28              0.877549             0.313922   
4 2024-02-04              0.637905             0.611007   
5 2024-02-11              0.790392             0.506448   
6 2024-02-18              0.524538             0.615129   
7 2024-02-25              0.765797             0.545606   
8 2024-03-03              0.737922             0.686245   
9 2024-03-10              0.578706             0.675499   

   sentiment_score_count  prob_positive_mean  prob_neutral_mean  \
0                     26            0.801286           0.196955   
1                     32            0.778525           0.220766   
2                     31            0.663601           0.330048   
3                     20            0.880783        

C:\Users\codes\AppData\Local\Temp\ipykernel_18128\873248904.py:10: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  aggregated = df.groupby(pd.Grouper(freq=freq)).agg({


In [ ]:
# Time-Platform aggregation for cross-analysis
weekly_platform = results_df.set_index('timestamp').groupby([
    pd.Grouper(freq='W'),
    'platform'
]).agg({
    'sentiment_score': 'mean',
    'text': 'count'
}).reset_index()

weekly_platform.columns = ['timestamp', 'platform', 'avg_sentiment', 'post_count']

print("Weekly Sentiment by Platform:")
print(weekly_platform.head(15))

# Calculate rolling averages (4-week moving average)
results_df_sorted = results_df.sort_values('timestamp')
results_df_sorted = results_df_sorted.set_index('timestamp')

rolling_sentiment = results_df_sorted['sentiment_score'].resample('D').mean().rolling(
    window=28, min_periods=7
).mean().reset_index()

rolling_sentiment.columns = ['timestamp', 'rolling_avg_28d']

print("\n" + "="*50)
print("28-Day Rolling Average Sentiment:")
print(rolling_sentiment.tail(10))

Weekly Sentiment by Platform:
    timestamp platform  avg_sentiment  post_count
0  2024-01-07  Twitter       0.799527          26
1  2024-01-14  Twitter       0.777816          32
2  2024-01-21  Twitter       0.657249          31
3  2024-01-28  Twitter       0.877549          20
4  2024-02-04  Twitter       0.637905          26
5  2024-02-11  Twitter       0.790392          24
6  2024-02-18  Twitter       0.524538          22
7  2024-02-25  Twitter       0.765797          26
8  2024-03-03  Twitter       0.737922          23
9  2024-03-10  Twitter       0.578706          27
10 2024-03-17  Twitter       0.735970          19
11 2024-03-24  Twitter       0.829228          29
12 2024-03-31  Twitter       0.928700          28
13 2024-04-07  Twitter       0.955613          23
14 2024-04-14  Twitter       0.834828          31

28-Day Rolling Average Sentiment:
     timestamp  rolling_avg_28d
358 2024-12-24         0.589719
359 2024-12-25         0.571593
360 2024-12-26         0.571541
361 202

## Phase 8: Time-Series Visualization

Create comprehensive time-series charts showing sentiment evolution.

### Quick Load: Skip Training and Load Pre-computed Results

If you've already completed training and want to skip directly to visualizations, run this cell to load the saved predictions.

In [14]:
# Quick Load: Load pre-computed predictions and prepare for visualization
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Load saved predictions
results_df = pd.read_csv("predictions_with_metadata.csv")
results_df['timestamp'] = pd.to_datetime(results_df['timestamp'])

print(f"✓ Loaded {len(results_df):,} predictions")
print(f"Date range: {results_df['timestamp'].min()} to {results_df['timestamp'].max()}")

# Recreate aggregation function
def aggregate_by_time(df, freq='W', score_col='sentiment_score'):
    """Aggregate sentiment by time period"""
    df = df.copy()
    df = df.set_index('timestamp')
    
    aggregated = df.groupby(pd.Grouper(freq=freq)).agg({
        score_col: ['mean', 'std', 'count'],
        'prob_positive': 'mean',
        'prob_neutral': 'mean',
        'prob_negative': 'mean',
        'confidence': 'mean'
    })
    
    aggregated.columns = ['_'.join(col).strip() for col in aggregated.columns.values]
    aggregated = aggregated.reset_index()
    
    return aggregated

# Generate weekly aggregations
weekly_sentiment = aggregate_by_time(results_df, freq='W')

# Generate weekly platform data
weekly_platform = results_df.set_index('timestamp').groupby([
    pd.Grouper(freq='W'),
    'platform'
]).agg({
    'sentiment_score': 'mean',
    'text': 'count'
}).reset_index()
weekly_platform.columns = ['timestamp', 'platform', 'avg_sentiment', 'post_count']

print(f"✓ Aggregations ready: {len(weekly_sentiment)} weeks")
print("\n✓ Ready for visualizations! Run the cells below.")

✓ Loaded 1,323 predictions
Date range: 2024-01-01 02:00:00 to 2025-01-02 09:00:00
✓ Aggregations ready: 53 weeks

✓ Ready for visualizations! Run the cells below.


In [15]:
# Visualize overall sentiment trends over time
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# 1. Weekly Sentiment Time Series with all sentiment types
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=weekly_sentiment['timestamp'],
    y=weekly_sentiment['prob_positive_mean'],
    mode='lines+markers',
    name='Positive',
    line=dict(color='green', width=2),
    marker=dict(size=6)
))

fig1.add_trace(go.Scatter(
    x=weekly_sentiment['timestamp'],
    y=weekly_sentiment['prob_neutral_mean'],
    mode='lines+markers',
    name='Neutral',
    line=dict(color='gray', width=2),
    marker=dict(size=6)
))

fig1.add_trace(go.Scatter(
    x=weekly_sentiment['timestamp'],
    y=weekly_sentiment['prob_negative_mean'],
    mode='lines+markers',
    name='Negative',
    line=dict(color='red', width=2),
    marker=dict(size=6)
))

fig1.update_layout(
    title='Weekly Sentiment Distribution Over Time',
    xaxis_title='Date',
    yaxis_title='Average Probability',
    hovermode='x unified',
    template='plotly_white',
    height=500,
    legend=dict(x=0.01, y=0.99)
)

fig1.show()

# 2. Sentiment Score Time Series (Composite -1 to +1)
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=weekly_sentiment['timestamp'],
    y=weekly_sentiment['sentiment_score_mean'],
    mode='lines+markers',
    name='Sentiment Score',
    line=dict(color='blue', width=3),
    marker=dict(size=8),
    fill='tozeroy',
    fillcolor='rgba(0,100,255,0.2)'
))

# Add zero line for reference
fig2.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)

fig2.update_layout(
    title='Weekly Sentiment Score Trend (-1: Negative, +1: Positive)',
    xaxis_title='Date',
    yaxis_title='Average Sentiment Score',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig2.show()

In [16]:
# 3. Platform Comparison Time Series
fig3 = px.line(
    weekly_platform,
    x='timestamp',
    y='avg_sentiment',
    color='platform',
    markers=True,
    title='Sentiment Trends by Platform',
    labels={'avg_sentiment': 'Average Sentiment Score', 'timestamp': 'Date'}
)

fig3.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.3)

fig3.update_layout(
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig3.show()

# 4. Volume and Sentiment Combined View
fig4 = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Sentiment Score Over Time', 'Post Volume Over Time'),
    vertical_spacing=0.12,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

# Sentiment trend
fig4.add_trace(
    go.Scatter(
        x=weekly_sentiment['timestamp'],
        y=weekly_sentiment['sentiment_score_mean'],
        mode='lines+markers',
        name='Sentiment Score',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1
)

# Volume trend
fig4.add_trace(
    go.Bar(
        x=weekly_sentiment['timestamp'],
        y=weekly_sentiment['sentiment_score_count'],
        name='Post Count',
        marker_color='lightblue'
    ),
    row=2, col=1
)

fig4.update_xaxes(title_text="Date", row=2, col=1)
fig4.update_yaxes(title_text="Sentiment Score", row=1, col=1)
fig4.update_yaxes(title_text="Number of Posts", row=2, col=1)

fig4.update_layout(
    title_text='Sentiment & Volume Analysis',
    showlegend=True,
    template='plotly_white',
    height=700
)

fig4.show()

## Phase 9: Geographic Heatmap

Visualize sentiment intensity by geolocation.

In [17]:
# Aggregate sentiment by geolocation
geo_sentiment = results_df.groupby('geolocation').agg({
    'sentiment_score': ['mean', 'std', 'count'],
    'prob_positive': 'mean',
    'prob_negative': 'mean',
    'prob_neutral': 'mean'
}).reset_index()

geo_sentiment.columns = ['geolocation', 'avg_sentiment', 'std_sentiment', 
                         'post_count', 'avg_positive', 'avg_negative', 'avg_neutral']

# Sort by average sentiment
geo_sentiment_sorted = geo_sentiment.sort_values('avg_sentiment', ascending=False)

print("Top 10 Most Positive Regions:")
print(geo_sentiment_sorted.head(10))

print("\n" + "="*50)
print("Top 10 Most Negative Regions:")
print(geo_sentiment_sorted.tail(10))

# Geographic Heatmap (Bar Chart representation)
fig5 = px.bar(
    geo_sentiment_sorted.head(20),
    x='geolocation',
    y='avg_sentiment',
    color='avg_sentiment',
    color_continuous_scale=['red', 'yellow', 'green'],
    color_continuous_midpoint=0,
    title='Top 20 Locations by Average Sentiment Score',
    labels={'avg_sentiment': 'Average Sentiment', 'geolocation': 'Location'},
    hover_data=['post_count', 'avg_positive', 'avg_negative']
)

fig5.update_layout(
    xaxis_tickangle=-45,
    template='plotly_white',
    height=500,
    showlegend=False
)

fig5.show()

# Alternative: Heatmap Matrix by Location and Time Period
# Prepare data for time-location heatmap
results_df['month'] = results_df['timestamp'].dt.to_period('M').astype(str)

geo_time_matrix = results_df.groupby(['geolocation', 'month'])['sentiment_score'].mean().reset_index()
geo_time_pivot = geo_time_matrix.pivot(index='geolocation', columns='month', values='sentiment_score')

# Filter to top locations by volume
top_locations = results_df['geolocation'].value_counts().head(15).index
geo_time_filtered = geo_time_pivot.loc[geo_time_pivot.index.isin(top_locations)]

fig6 = px.imshow(
    geo_time_filtered,
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title='Sentiment Heatmap: Top Locations Over Time',
    labels=dict(x="Month", y="Location", color="Sentiment Score"),
    aspect='auto'
)

fig6.update_layout(
    height=600,
    xaxis_tickangle=-45
)

fig6.show()

Top 10 Most Positive Regions:
  geolocation  avg_sentiment  std_sentiment  post_count  avg_positive  \
0     Unknown       0.767925       0.528702        1323      0.830147   

   avg_negative  avg_neutral  
0      0.062221     0.107632  

Top 10 Most Negative Regions:
  geolocation  avg_sentiment  std_sentiment  post_count  avg_positive  \
0     Unknown       0.767925       0.528702        1323      0.830147   

   avg_negative  avg_neutral  
0      0.062221     0.107632  


## Phase 10: Advanced Analytics & Insights

Statistical analysis and key findings from sentiment data.

In [18]:
# Advanced Statistical Analysis
print("=" * 70)
print("SENTIMENT ANALYSIS SUMMARY")
print("=" * 70)

# Overall statistics
print(f"\nDataset Overview:")
print(f"Total Predictions: {len(results_df):,}")
print(f"Date Range: {results_df['timestamp'].min()} to {results_df['timestamp'].max()}")
print(f"\nOverall Sentiment Distribution:")
print(f"  Positive: {(results_df['predicted_label'] == 'positive').sum():,} ({(results_df['predicted_label'] == 'positive').mean()*100:.1f}%)")
print(f"  Neutral:  {(results_df['predicted_label'] == 'neutral').sum():,} ({(results_df['predicted_label'] == 'neutral').mean()*100:.1f}%)")
print(f"  Negative: {(results_df['predicted_label'] == 'negative').sum():,} ({(results_df['predicted_label'] == 'negative').mean()*100:.1f}%)")

print(f"\nSentiment Score Statistics:")
print(f"  Mean: {results_df['sentiment_score'].mean():.4f}")
print(f"  Median: {results_df['sentiment_score'].median():.4f}")
print(f"  Std Dev: {results_df['sentiment_score'].std():.4f}")
print(f"  Range: [{results_df['sentiment_score'].min():.4f}, {results_df['sentiment_score'].max():.4f}]")

print(f"\nConfidence Metrics:")
print(f"  Average Confidence: {results_df['confidence'].mean():.4f}")
print(f"  High Confidence (>0.9): {(results_df['confidence'] > 0.9).sum():,} ({(results_df['confidence'] > 0.9).mean()*100:.1f}%)")
print(f"  Low Confidence (<0.6): {(results_df['confidence'] < 0.6).sum():,} ({(results_df['confidence'] < 0.6).mean()*100:.1f}%)")

print("\n" + "=" * 70)

SENTIMENT ANALYSIS SUMMARY

Dataset Overview:
Total Predictions: 1,323
Date Range: 2024-01-01 02:00:00 to 2025-01-02 09:00:00

Overall Sentiment Distribution:
  Positive: 0 (0.0%)
  Neutral:  0 (0.0%)
  Negative: 0 (0.0%)

Sentiment Score Statistics:
  Mean: 0.7679
  Median: 0.9991
  Std Dev: 0.5287
  Range: [-0.9979, 0.9993]

Confidence Metrics:
  Average Confidence: 0.9834
  High Confidence (>0.9): 1,259 (95.2%)
  Low Confidence (<0.6): 10 (0.8%)



---

## Summary: MLOps Pipeline Complete ✓

**Completed Tasks:**

1. ✓ **Fine-Tuning**: RoBERTa-base fine-tuned on sentiment data with robust evaluation (F1, Accuracy)
2. ✓ **Model Evaluation**: Comprehensive metrics including confusion matrix and per-class F1 scores
3. ✓ **Sentiment Classification**: Generated sentiment labels and confidence scores for entire test set
4. ✓ **Temporal Aggregation**: Weekly/Monthly rolling averages grouped by time and platform
5. ✓ **Visualization**: Interactive Plotly charts showing sentiment evolution, platform comparison, and volume trends
6. ✓ **Geographic Analysis**: Heatmap showing sentiment intensity by location
7. ✓ **Statistical Analysis**: Comprehensive trend analysis, volatility metrics, and insights

**Output Files:**
- `models/roberta-environmental-sentiment-best/` - Fine-tuned model checkpoint
- `predictions_with_metadata.csv` - Full predictions with timestamps, platforms, geolocation

**Key Insights Available:**
- Temporal trends and seasonality patterns
- Platform-specific sentiment differences
- Geographic sentiment distribution
- Statistical summary and volatility analysis

In [2]:
# Install all required dependencies including nbformat for Plotly visualization
import sys
!{sys.executable} -m pip install -q transformers datasets torch scikit-learn pandas plotly prophet kaleido nbformat>=4.2.0

In [4]:
# Verify nbformat installation and restart kernel if needed
try:
    import nbformat
    print(f"✓ nbformat version: {nbformat.__version__}")
    print("✓ Plotly visualizations should work now")
except ImportError:
    print("⚠ nbformat not found. Please restart the kernel (Ctrl+Shift+P → 'Restart Kernel')")
    print("⚠ Then re-run the cells from Phase 1 onwards")

✓ nbformat version: 5.10.4
✓ Plotly visualizations should work now


## Generate Predictions for Analytics Dashboard

In [ ]:
import pandas as pd
import torch
import numpy as np

# 1. Run predictions on the test/validation set
print("Generating predictions for Analytics Dashboard...")
predictions_output = trainer.predict(val_ds) # 'val_ds' or 'test_ds' depending on your split name

# 2. Convert raw logits to probabilities (Confidence scores)
# RoBERTa outputs "logits" (raw numbers), we need 0-1 probabilities
logits = torch.tensor(predictions_output.predictions)
probs = torch.nn.functional.softmax(logits, dim=-1)

# 3. Get the winning label (0, 1, or 2)
preds = np.argmax(predictions_output.predictions, axis=-1)

# 4. Create the Dataframe
# We convert the Hugging Face dataset back to a Pandas DataFrame
df_out = pd.DataFrame(val_ds) 

# Add the new columns your Analytics Notebook needs
df_out['predicted_label'] = preds
df_out['confidence'] = probs.max(dim=1).values.numpy() # The highest score
df_out['prob_negative'] = probs[:, 0].numpy()
df_out['prob_neutral'] = probs[:, 1].numpy()
df_out['prob_positive'] = probs[:, 2].numpy()

# Map numeric labels to words (Optional, but helps readability)
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df_out['predicted_sentiment'] = df_out['predicted_label'].map(label_map)

# 5. Save the bridge file
output_file = "predictions_with_metadata.csv"
df_out.to_csv(output_file, index=False)

print(f"✅ Success! Predictions saved to '{output_file}'.")
print(f"You can now open 'Advanced_Sentiment_Analytics.ipynb' and run it.")
print(df_out.head())